# Bloc 3 — Calculateur d'accord entre annotateurs (20 min)

**Situation.** Trois personnes du support étiquettent les **15 mêmes messages** chacune de son côté. Elles ne sont d'accord que sur une partie ; sur *« internet coupe tout le temps, je vais finir par résilier »*, l'une met *technique*, l'autre *résiliation*. Si les humains ne s'accordent pas, par rapport à qui mesure-t-on le modèle ?

C'est votre tour : remplissez la fiche du bloc 3 **seul et en silence**, puis on compare.

Chaque participant étiquette les **15 mêmes messages** (fiche `docs/fiches/fiche_bloc3_annotation`). On saisit les réponses dans `dataset/annotations_atelier.csv` (une colonne par personne : A1…A10, valeurs `facturation`, `technique`, `resiliation`, `livraison`, `autre`).

Tant que ce fichier est vide, le notebook utilise `dataset/annotations_exemple_SIMULEES.csv` : **des annotations simulées**, pas de vraies personnes. Elles servent seulement à montrer les calculs.

**Pourquoi c'est important ?** Si les humains ne sont pas d'accord entre eux, aucun modèle ne peut être « juste » sur ces cas, et l'exactitude mesurée a un plafond.

In [ ]:
import itertools
from collections import Counter
from pathlib import Path
import pandas as pd

DATA = Path("..") / "dataset"
CLASSES = ["facturation", "technique", "resiliation", "livraison", "autre"]
reel = pd.read_csv(DATA / "annotations_atelier.csv", dtype=str).fillna("")
cols = [c for c in reel.columns if c.startswith("A") and (reel[c] != "").all()]
if len(cols) >= 2:
    annot, source = reel, "annotations du groupe"
else:
    annot, source = pd.read_csv(DATA / "annotations_exemple_SIMULEES.csv", dtype=str), "annotations SIMULÉES (exemple)"
    cols = [c for c in annot.columns if c.startswith("A")]
messages = pd.read_csv(DATA / "labeling_15.csv")
print(f"Source : {source} — {len(cols)} annotateurs, {len(annot)} messages")
annot[["numero"] + cols]

## 1. Accord brut (en %)
Pour chaque paire de personnes : part des messages où elles ont mis la même étiquette. On fait ensuite la moyenne sur toutes les paires.

In [ ]:
import sys; sys.path.insert(0, "..")
from atelier import accord, kappa_cohen, kappa_fleiss   # le code est dans atelier/mesures.py

paires = {(i, j): accord(annot[i], annot[j]) for i, j in itertools.combinations(cols, 2)}
print(f"Accord brut moyen entre deux personnes : {100 * sum(paires.values()) / len(paires):.1f} %")

**Question avant la suite** : deux personnes qui cocheraient **au hasard** une des 5 cases, à quelle fréquence tomberaient-elles d'accord ? Et si tout le monde répond « technique » parce que c'est le service le plus fréquent ?
*Réponse attendue* : environ 1 fois sur 5 au hasard, et beaucoup plus si les réponses sont concentrées. Donc 65 % d'accord brut, ce n'est pas 65 % de « vrai » accord.

## 2. Kappa de Cohen (deux personnes)
L'accord brut est flatteur : deux personnes qui répondent au hasard tombent parfois d'accord. Le **kappa** retire cet accord « par chance » :
κ = (accord observé − accord attendu par hasard) / (1 − accord attendu).
Repères usuels (Landis & Koch) : < 0,2 faible · 0,2–0,4 passable · 0,4–0,6 modéré · 0,6–0,8 bon · > 0,8 très bon.

In [ ]:
A, B = cols[0], cols[1]
print(f"Cohen κ entre {A} et {B} : {kappa_cohen(annot[A], annot[B]):.3f}  (accord brut {100*accord(annot[A], annot[B]):.0f} %)")
mat = pd.DataFrame([[kappa_cohen(annot[i], annot[j]) for j in cols] for i in cols], index=cols, columns=cols).round(2)
mat

## 3. Kappa de Fleiss (tout le groupe d'un coup)

In [ ]:
kf, accord_par_message = kappa_fleiss(annot[cols])
print(f"Fleiss κ du groupe : {kf:.3f}")

## 4. Sur quels messages sommes-nous en désaccord ?

In [ ]:
corrige = pd.read_csv(DATA / "labeling_15_corrige.csv").fillna("")
vue = messages.copy()
vue["majorite"] = [Counter(r).most_common(1)[0][0] for r in annot[cols].values]
vue["accord_%"] = [round(100 * Counter(r).most_common(1)[0][1] / len(cols)) for r in annot[cols].values]
vue["reference"] = corrige["label"]
vue["acceptables"] = corrige["labels_acceptables"]
vue["difficulte"] = corrige["difficulte"]
vue.sort_values("accord_%")[["numero", "message", "majorite", "accord_%", "reference", "acceptables", "difficulte"]]

**À discuter**
- Les désaccords tombent-ils sur les messages « multi » ou « ambigus » ? Ce sont exactement ceux que la politique finale devra **envoyer à un humain**.
- Faut-il une règle d'annotation (« si le client parle d'argent ET de panne, on prend ce qu'il demande en premier ») ? Écrivez-la : elle deviendra une phrase dans vos options.

## 5. Dev vs test scellé
**Question** : si je réécris mes options jusqu'à obtenir 100 % sur ces messages-là, en les regardant à chaque fois, que vaut ce 100 % pour les messages de lundi prochain ?
*Réponse attendue* : pas grand-chose : on a appris *ces* messages. Il faut un paquet qu'on ne regarde pas. Ce paquet a un nom :
- **dev** (`dataset/dev.csv`, 63 messages) : on a le droit de regarder, de se tromper, de réécrire les options. C'est l'atelier de travail.
- **test scellé** (`dataset/test_scelle.csv`, 77 messages) : on ne l'ouvre **qu'une fois**, à la fin, avec des options **figées**. Sinon on finit par ajuster les options au test, et le score ment.
- *Mode rapide* (PC sans GPU Apple ni NVIDIA) : les notebooks utilisent `dev_rapide.csv` (20) et `test_scelle_rapide.csv` (25), tirés de façon stratifiée dans les mêmes fichiers.

### Bonus : wyat comme 11ᵉ annotateur

In [ ]:
RUN_WYAT = True   # passez à False si le modèle n'est pas installé
if RUN_WYAT:
    import atelier as at   # (déjà dans le chemin, voir plus haut)
    modele = at.charger_modele()
    decisions = [at.decider(modele, m) for m in messages["message"]]
    vue["wyat"] = [d.selected for d in decisions]
    vue["wyat_marge"] = [round(d.probability_margin, 2) for d in decisions]
    print("wyat = référence :", (vue["wyat"] == vue["reference"]).sum(), "/ 15 ;",
          "wyat = majorité du groupe :", (vue["wyat"] == vue["majorite"]).sum(), "/ 15")
    display(vue[["numero", "message", "majorite", "reference", "wyat", "wyat_marge"]])

---
**Étape 3, point de contrôle** (`missions/3_accord/MISSION.md`) : le même objectif, à atteindre comme vous voulez (ce notebook, la commande `atelier`, ou un script).
Vérification sur dev : `atelier verifier 3` dans un terminal, ou la cellule ci-dessous.

In [ ]:
from pathlib import Path
from atelier.missions import preparer, verifier
preparer(Path("../travail"))   # crée travail/ avec les fichiers de départ s'ils manquent (n'écrase rien)
print(verifier(3, globals().get("modele"), Path("../travail")).texte())   # réutilise le modèle déjà chargé